# 🛒 RFM Customer Segmentation Analysis
**Dataset:** Turkish E-Commerce Customer Behavior & Sales (2023–2024)
**Author:** Ilker Trker | github.com/ilkertrker


## 1 — Setup & Data Loading

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Load dataset ──────────────────────────────────────────────────────────────
# Download from: kaggle.com/datasets/umuttuygurr/e-commerce-customer-behavior-and-sales-analysis-tr
df = pd.read_csv('data/ecommerce_customer_behavior_dataset_v2.csv')
df['Date'] = pd.to_datetime(df['Date'])

print(f'Rows: {len(df):,}')
print(f'Unique customers: {df["Customer_ID"].nunique():,}')
print(f'Date range: {df["Date"].min().date()} → {df["Date"].max().date()}')
print(f'Total revenue: ₺{df["Total_Amount"].sum():,.0f}')
df.head()

## 2 — Exploratory Data Analysis (EDA)

In [ ]:
# Category breakdown
print('--- Revenue by Category ---')
print(df.groupby('Product_Category')['Total_Amount'].sum().sort_values(ascending=False).apply(lambda x: f'₺{x:,.0f}'))

print('\n--- Revenue by City ---')
print(df.groupby('City')['Total_Amount'].sum().sort_values(ascending=False).apply(lambda x: f'₺{x:,.0f}'))

print('\n--- Payment Methods ---')
print(df['Payment_Method'].value_counts())

## 3 — RFM Calculation

In [ ]:
# Snapshot date = 1 day after last order
snapshot_date = df['Date'].max() + pd.Timedelta(days=1)

rfm = df.groupby('Customer_ID').agg(
    Recency   = ('Date',         lambda x: (snapshot_date - x.max()).days),
    Frequency = ('Order_ID',     'count'),
    Monetary  = ('Total_Amount', 'sum')
).reset_index()

# Score each dimension 1-5
rfm['R_Score'] = pd.qcut(rfm['Recency'],   5, labels=[5,4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'],  5, labels=[1,2,3,4,5]).astype(int)
rfm['FM_avg']  = (rfm['F_Score'] + rfm['M_Score']) / 2

rfm.describe().round(1)

## 4 — Customer Segmentation

In [ ]:
def assign_segment(r, fm):
    if r >= 4 and fm >= 4:   return 'Champions'
    elif r >= 3 and fm >= 3: return 'Loyal Customers'
    elif r >= 3 and fm >= 1: return 'Potential Loyalists'
    elif r >= 2 and fm >= 3: return 'At Risk'
    elif r >= 2 and fm >= 1: return 'Need Attention'
    elif r <= 1 and fm >= 2: return 'Lost'
    else:                    return 'Promising'

rfm['Segment'] = rfm.apply(lambda row: assign_segment(row['R_Score'], row['FM_avg']), axis=1)

seg_stats = rfm.groupby('Segment').agg(
    Count         = ('Customer_ID', 'count'),
    Total_Revenue = ('Monetary',    'sum'),
    Avg_Spend     = ('Monetary',    'mean'),
    Avg_Frequency = ('Frequency',   'mean'),
    Avg_Recency   = ('Recency',     'mean')
).reset_index().sort_values('Total_Revenue', ascending=False)

seg_stats['Revenue_Share']  = (seg_stats['Total_Revenue']  / seg_stats['Total_Revenue'].sum()  * 100).round(1)
seg_stats['Customer_Share'] = (seg_stats['Count'] / seg_stats['Count'].sum() * 100).round(1)

print(seg_stats[['Segment','Count','Customer_Share','Revenue_Share','Avg_Spend','Avg_Frequency']].to_string(index=False))

## 5 — Visualizations

In [ ]:
SEGMENT_COLORS = {
    'Champions':'#534AB7', 'Loyal Customers':'#1D9E75',
    'At Risk':'#BA7517',   'Lost':'#E24B4A',
    'Potential Loyalists':'#378ADD', 'Need Attention':'#D85A30',
    'Promising':'#888780'
}
BG='#FAFAF8'; TEXT='#2C2C2A'; MUTED='#73726c'

plt.rcParams.update({
    'font.family':'DejaVu Sans','figure.facecolor':BG,'axes.facecolor':'#FFFFFF',
    'axes.spines.top':False,'axes.spines.right':False,'axes.spines.left':False,
    'axes.edgecolor':'#D3D1C7','axes.labelcolor':MUTED,
    'xtick.color':MUTED,'ytick.color':MUTED,
    'grid.color':'#E8E6DF','grid.linewidth':0.6,'text.color':TEXT,'font.size':11,
})

In [ ]:
# Plot 1: Segment distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.5))
fig.patch.set_facecolor(BG)
colors = [SEGMENT_COLORS.get(s,'#888780') for s in seg_stats['Segment']]
wedges, texts, autotexts = ax1.pie(
    seg_stats['Count'], colors=colors, autopct='%1.0f%%', pctdistance=0.75,
    wedgeprops=dict(width=0.55, edgecolor=BG, linewidth=2), startangle=90
)
for at in autotexts: at.set_fontsize(9); at.set_color(TEXT)
ax1.text(0, 0, '5,000\ncustomers', ha='center', va='center', fontsize=11, fontweight='bold', color=TEXT)
ax1.set_title('Customer distribution by segment', fontsize=13, fontweight='bold', pad=14, loc='left')
handles = [mpatches.Patch(color=SEGMENT_COLORS.get(s,'#888780'), label=s) for s in seg_stats['Segment']]
ax1.legend(handles=handles, loc='lower center', frameon=False, fontsize=9, ncol=2, bbox_to_anchor=(0.5,-0.12))
rev_s = seg_stats.sort_values('Revenue_Share')
colors2 = [SEGMENT_COLORS.get(s,'#888780') for s in rev_s['Segment']]
bars = ax2.barh(rev_s['Segment'], rev_s['Revenue_Share'], color=colors2, height=0.6)
for bar, val in zip(bars, rev_s['Revenue_Share']):
    ax2.text(val+0.3, bar.get_y()+bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=9.5, color=MUTED)
ax2.set_xlabel('Share of total revenue', fontsize=10, color=MUTED)
ax2.grid(axis='x', linestyle='--', alpha=0.7); ax2.set_xlim(0,52)
ax2.set_title('Revenue share by segment', fontsize=13, fontweight='bold', pad=14, loc='left')
plt.tight_layout()
plt.savefig('plots/rfm_plot1_segments.png', dpi=180, bbox_inches='tight', facecolor=BG)
plt.show(); print('✓ Plot 1 saved')

In [ ]:
# Plot 2: Revenue vs Customer share bubble
fig, ax = plt.subplots(figsize=(10, 5.5)); fig.patch.set_facecolor(BG)
for _, row in seg_stats.iterrows():
    c = SEGMENT_COLORS.get(row['Segment'],'#888780')
    ax.scatter(row['Customer_Share'], row['Revenue_Share'], s=row['Avg_Spend']/10, color=c, alpha=0.85, zorder=3)
    offset = 5 if row['Segment'] not in ['At Risk','Lost'] else -12
    ax.annotate(row['Segment'], (row['Customer_Share'], row['Revenue_Share']),
                xytext=(0, offset), textcoords='offset points', ha='center', fontsize=9.5, color=TEXT)
ax.axline((0,0), slope=1, color='#D3D1C7', linestyle='--', linewidth=1, label='Equal share line')
ax.set_xlabel('% of customers', fontsize=10, color=MUTED)
ax.set_ylabel('% of revenue', fontsize=10, color=MUTED)
ax.grid(linestyle='--', alpha=0.5); ax.legend(frameon=False, fontsize=9)
ax.set_title('Revenue vs customer share  (bubble = avg spend)', fontsize=13, fontweight='bold', pad=14, loc='left')
plt.tight_layout()
plt.savefig('plots/rfm_plot2_revenue_vs_customers.png', dpi=180, bbox_inches='tight', facecolor=BG)
plt.show(); print('✓ Plot 2 saved')

In [ ]:
# Plot 3: Monthly revenue trend
df['YearMonth'] = df['Date'].dt.to_period('M')
monthly = df.groupby('YearMonth')['Total_Amount'].sum().reset_index()
monthly['YearMonth_str'] = monthly['YearMonth'].astype(str)
x = np.arange(len(monthly))
fig, ax = plt.subplots(figsize=(12, 4.5)); fig.patch.set_facecolor(BG)
ax.fill_between(x, monthly['Total_Amount'], alpha=0.08, color='#534AB7')
ax.plot(x, monthly['Total_Amount'], color='#534AB7', linewidth=2.2, zorder=3)
ax.scatter(x, monthly['Total_Amount'], color='#534AB7', s=40, zorder=4)
ax.set_xticks(x[::2]); ax.set_xticklabels(monthly['YearMonth_str'].iloc[::2], rotation=30, ha='right', fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'₺{v/1000:.0f}K'))
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.set_title('Monthly revenue trend — Jan 2023 to Mar 2024', fontsize=13, fontweight='bold', pad=14, loc='left')
plt.tight_layout()
plt.savefig('plots/rfm_plot3_monthly_revenue.png', dpi=180, bbox_inches='tight', facecolor=BG)
plt.show(); print('✓ Plot 3 saved')

In [ ]:
# Plot 4: Category revenue by segment
df2 = df.merge(rfm[['Customer_ID','Segment']], on='Customer_ID')
top4 = ['Champions','Loyal Customers','At Risk','Lost']
cat_seg = df2[df2['Segment'].isin(top4)].groupby(['Product_Category','Segment'])['Total_Amount'].sum().unstack(fill_value=0)
cat_seg = cat_seg.reindex(columns=top4)
cat_seg['Total'] = cat_seg.sum(axis=1)
cat_seg = cat_seg.sort_values('Total', ascending=True).drop(columns='Total')
fig, ax = plt.subplots(figsize=(11, 5.5)); fig.patch.set_facecolor(BG)
bottom = np.zeros(len(cat_seg))
for seg in top4:
    c = SEGMENT_COLORS.get(seg,'#888780')
    ax.barh(cat_seg.index, cat_seg[seg].values, left=bottom, color=c, label=seg, height=0.6)
    bottom += cat_seg[seg].values
ax.set_xlabel('Total revenue (₺)', fontsize=10, color=MUTED)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'₺{v/1000:.0f}K'))
ax.grid(axis='x', linestyle='--', alpha=0.7)
handles = [mpatches.Patch(color=SEGMENT_COLORS[s], label=s) for s in top4]
ax.legend(handles=handles, frameon=False, fontsize=9.5, loc='lower right')
ax.set_title('Revenue by product category and customer segment', fontsize=13, fontweight='bold', pad=14, loc='left')
plt.tight_layout()
plt.savefig('plots/rfm_plot4_category_segments.png', dpi=180, bbox_inches='tight', facecolor=BG)
plt.show(); print('✓ Plot 4 saved')

In [ ]:
# Plot 5: RFM scatter
fig, ax = plt.subplots(figsize=(10, 5.5)); fig.patch.set_facecolor(BG)
for seg, group in rfm.groupby('Segment'):
    c = SEGMENT_COLORS.get(seg,'#888780')
    ax.scatter(group['Recency'], group['Monetary'], color=c, alpha=0.45, s=18, label=seg, zorder=2)
ax.set_xlabel('Recency (days since last purchase)', fontsize=10, color=MUTED)
ax.set_ylabel('Monetary value (₺)', fontsize=10, color=MUTED)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'₺{v/1000:.0f}K'))
ax.grid(linestyle='--', alpha=0.5)
handles = [mpatches.Patch(color=SEGMENT_COLORS.get(s,'#888780'), label=s) for s in seg_stats['Segment']]
ax.legend(handles=handles, frameon=False, fontsize=9, ncol=2)
ax.set_title('RFM scatter — recency vs total spend per customer', fontsize=13, fontweight='bold', pad=14, loc='left')
plt.tight_layout()
plt.savefig('plots/rfm_plot5_rfm_scatter.png', dpi=180, bbox_inches='tight', facecolor=BG)
plt.show(); print('✓ Plot 5 saved')

## 6 — Key Findings Summary

In [ ]:
print('=' * 55)
print('KEY FINDINGS — RFM CUSTOMER SEGMENTATION')
print('=' * 55)
for _, row in seg_stats.iterrows():
    print(f"\n{row['Segment']}")
    print(f"  Customers : {row['Count']:,} ({row['Customer_Share']:.1f}%)")
    print(f"  Revenue   : ₺{row['Total_Revenue']:,.0f} ({row['Revenue_Share']:.1f}%)")
    print(f"  Avg spend : ₺{row['Avg_Spend']:,.0f}")
    print(f"  Avg orders: {row['Avg_Frequency']:.1f}")